## Module 3-3o Using the Google Maps API in Accounting Research

In this (optional) exercise, we will explore Google's Python clients for the Google Maps Platform — `googlemaps` for geocoding, and the `google-maps-routing` and `google-maps-places` clients for directions, distances, and place search — converting addresses to and from geographic coordinates, and computing distances/travel times and nearby places once you have coordinates in hand.

### 1. Setup
- Install the required packages:

```bash
uv add -U googlemaps google-maps-routing google-maps-places
```

- Get your Google Maps API key:

1. Go to [Google Cloud Console](https://console.cloud.google.com/).
2. Create a project (or select an existing one).
3. Enable the **Google Maps Platform APIs and Services**.
4. Go to **Credentials** and generate an **API Key**. Enabling at least the following services:

   - Geocoding API
   - Routes API (covers directions and distance/duration calculations)
   - Places API (New)


5. Save your API key in `.env` file.

---
**Best Practices**:

- **Rate limits**: Watch for usage quotas in your Google Cloud Console.
- **Error handling**: Handle exceptions for failed requests.
- **Security**: Never expose your API key publicly.

---

**Resources**:

- [`googlemaps` package documentation](https://github.com/googlemaps/google-maps-services-python) — used for Geocoding (Section 2)
- [`google-maps-routing` client documentation](https://googleapis.dev/python/routing/latest/) — used for Directions/Distance Matrix (Section 3)
- [`google-maps-places` client documentation](https://googleapis.dev/python/places/latest/) — used for Places (Section 4)
- [Google Maps Platform Documentation](https://developers.google.com/maps/documentation)
- [API Pricing](https://cloud.google.com/maps-platform/pricing/)


### 2. Basic usage of the client

In [ ]:
import googlemaps
from google.maps import routing_v2
from google.maps import places_v1
from dotenv import load_dotenv
load_dotenv()
import os

In [ ]:
gmaps = googlemaps.Client(key = os.getenv('GOOGLE_MAPS_API'))

# Routes API and Places API (New) clients, authenticated with the same API key
routes_client = routing_v2.RoutesClient(client_options={"api_key": os.getenv('GOOGLE_MAPS_API')})
places_client = places_v1.PlacesClient(client_options={"api_key": os.getenv('GOOGLE_MAPS_API')})

#### 2.1. Geocoding (address → lat/lng)

In [ ]:
geocode_result = gmaps.geocode("255 George St, Sydney NSW 2000, Australia")
print(geocode_result[0]['geometry']['location'])

In [ ]:
def get_lat_long(address: str) -> tuple:
    geocode_result = gmaps.geocode(address)
    if geocode_result:
        lat = geocode_result[0].get('geometry').get('location').get('lat')
        long = geocode_result[0].get('geometry').get('location').get('lng')
        return lat, long
    return None, None

#### 2.2. Reverse Geocoding (lat/lng → address)


In [ ]:
reverse_geocode_result = gmaps.reverse_geocode((37.4221, -122.0841))
print(reverse_geocode_result[0]['formatted_address'])

### 3. Distance API

Sections 3 and 4 use the **Routes API** and **Places API (New)** rather than the older `googlemaps` convenience methods (`directions()`, `distance_matrix()`, `places_nearby()`, etc.) — those wrap Google's *legacy* APIs, which new Cloud projects can no longer enable (see the note in Setup above). The code below uses the official `google-maps-routing` and `google-maps-places` client libraries instead, talking to the current APIs directly.

#### 3.1. Directions

In [ ]:
directions_request = routing_v2.ComputeRoutesRequest(
    origin=routing_v2.Waypoint(address="255 George St, Sydney NSW 2000, Australia"),
    destination=routing_v2.Waypoint(address="159-175 Church St, Parramatta NSW 2150, Australia"),
    travel_mode=routing_v2.RouteTravelMode.DRIVE,
)

directions_response = routes_client.compute_routes(
    request=directions_request,
    metadata=[("x-goog-fieldmask", "routes.legs.steps.navigationInstruction.instructions")],
)

# Print step-by-step directions
for step in directions_response.routes[0].legs[0].steps:
    print(step.navigation_instruction.instructions)

#### 3.2. Distance Matrix

In [ ]:
matrix_request = routing_v2.ComputeRouteMatrixRequest(
    origins=[
        routing_v2.RouteMatrixOrigin(waypoint=routing_v2.Waypoint(address="New York, NY")),
    ],
    destinations=[
        routing_v2.RouteMatrixDestination(waypoint=routing_v2.Waypoint(address="Philadelphia, PA")),
        routing_v2.RouteMatrixDestination(waypoint=routing_v2.Waypoint(address="Boston, MA")),
    ],
    travel_mode=routing_v2.RouteTravelMode.DRIVE,
)

# compute_route_matrix returns one RouteMatrixElement per origin-destination pair,
# with raw distance_meters/duration instead of the legacy API's pre-formatted text
for element in routes_client.compute_route_matrix(
    request=matrix_request,
    metadata=[("x-goog-fieldmask", "originIndex,destinationIndex,distanceMeters,duration,condition")],
):
    distance_km = element.distance_meters / 1000
    duration_min = element.duration.seconds / 60
    print(f"Distance: {distance_km:.1f} km, Duration: {duration_min:.0f} mins")

### 4. Other functions

#### 4.1. Places API (Find nearby places)

In [ ]:
nearby_request = places_v1.SearchNearbyRequest(
    location_restriction={
        "circle": {
            "center": {"latitude": 37.4221, "longitude": -122.0841},
            "radius": 1000.0,
        }
    },
    included_types=["restaurant"],
)

nearby_response = places_client.search_nearby(
    request=nearby_request,
    metadata=[("x-goog-fieldmask", "places.displayName,places.formattedAddress")],
)

for place in nearby_response.places:
    print(place.display_name.text, "-", place.formatted_address or "N/A")

#### 4.2. Autocomplete and Place Details

In [ ]:
# Autocomplete example
autocomplete_response = places_client.autocomplete_places(
    request=places_v1.AutocompletePlacesRequest(input="Starb"),
    metadata=[("x-goog-fieldmask", "suggestions.placePrediction.text,suggestions.placePrediction.placeId")],
)
print([s.place_prediction.text.text for s in autocomplete_response.suggestions])

In [ ]:
# Get place details
place_id = autocomplete_response.suggestions[0].place_prediction.place_id
details = places_client.get_place(
    request=places_v1.GetPlaceRequest(name=f"places/{place_id}"),
    metadata=[("x-goog-fieldmask", "displayName,formattedAddress")],
)
print(details.display_name.text, details.formatted_address)